# Chapter 5: CAPM and Beta Estimation

This notebook implements the Capital Asset Pricing Model (CAPM) and estimates beta using real market data.

In [ ]:
import sys
sys.path.append('../')

import numpy as np
import matplotlib.pyplot as plt

from src.chapter2_returns import simple_return, expected_return
from src.chapter5_capm import beta, capm_expected_return
from src.data_utils import download_and_save_prices

## Download Market and Asset Data

We use the SMI index (`^SSMI`) as the market proxy and UBS as the asset.

In [ ]:
start_date = '2022-01-01'
end_date = '2024-01-01'

prices_market, _ = download_and_save_prices('^SSMI', start_date, end_date, data_dir='../data')
prices_ubs, _ = download_and_save_prices('UBS', start_date, end_date, data_dir='../data')

# Align lengths
min_len = min(len(prices_market), len(prices_ubs))
prices_market = prices_market[:min_len]
prices_ubs = prices_ubs[:min_len]

print(f"Number of trading days: {min_len}")

## Calculate Returns

In [ ]:
ret_market = simple_return(prices_market)
ret_ubs = simple_return(prices_ubs)

print(f"Market (SMI) — daily mean return: {expected_return(ret_market)*100:.4f}%")
print(f"UBS          — daily mean return: {expected_return(ret_ubs)*100:.4f}%")

## Estimate Beta

Beta measures systematic (market) risk:

$$\beta_i = \frac{\text{Cov}(r_i, r_m)}{\text{Var}(r_m)}$$

- $\beta = 1$: asset moves in line with the market
- $\beta > 1$: asset is more volatile than the market
- $\beta < 1$: asset is less volatile than the market

In [ ]:
ubs_beta = beta(ret_ubs, ret_market)

print(f"UBS Beta (vs SMI): {ubs_beta:.4f}")
if ubs_beta > 1:
    print("→ UBS is more volatile than the SMI (amplifies market moves)")
elif ubs_beta < 1:
    print("→ UBS is less volatile than the SMI (dampens market moves)")
else:
    print("→ UBS moves in line with the SMI")

## CAPM Expected Return

CAPM predicts the required return for bearing systematic risk:

$$E[r_i] = r_f + \beta_i \cdot (E[r_m] - r_f)$$

where $(E[r_m] - r_f)$ is the **market risk premium**.

In [ ]:
# Annualized inputs
risk_free_rate = 0.015        # Swiss 10-year govt bond approx 1.5%
market_return = expected_return(ret_market) * 252

capm_ret = capm_expected_return(risk_free_rate, ubs_beta, market_return)
actual_ret = expected_return(ret_ubs) * 252

print(f"Inputs:")
print(f"  Risk-free rate:       {risk_free_rate*100:.2f}%")
print(f"  Market return (SMI):  {market_return*100:.2f}%")
print(f"  Market risk premium:  {(market_return - risk_free_rate)*100:.2f}%")
print(f"  UBS Beta:             {ubs_beta:.4f}")
print(f"\nResults:")
print(f"  CAPM expected return: {capm_ret*100:.2f}%")
print(f"  Actual return:        {actual_ret*100:.2f}%")
print(f"  Alpha (actual - CAPM): {(actual_ret - capm_ret)*100:.2f}%")

## Security Market Line (SML)

The SML plots expected return as a function of beta for all assets in equilibrium.

In [ ]:
beta_range = np.linspace(0, 2, 100)
sml_returns = [capm_expected_return(risk_free_rate, b, market_return) for b in beta_range]

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(beta_range, np.array(sml_returns) * 100, color='steelblue', linewidth=2, label='SML')
ax.scatter([ubs_beta], [capm_ret * 100], color='orange', zorder=5, s=100, label=f'CAPM E[r] UBS = {capm_ret*100:.1f}%')
ax.scatter([ubs_beta], [actual_ret * 100], color='red', zorder=5, s=100, marker='D', label=f'Actual E[r] UBS = {actual_ret*100:.1f}%')
ax.axhline(y=risk_free_rate * 100, color='gray', linestyle='--', linewidth=1, label=f'Risk-free rate = {risk_free_rate*100:.1f}%')

ax.set_xlabel('Beta')
ax.set_ylabel('Expected Return (%)')
ax.set_title('Security Market Line (CAPM)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Beta via Scatter Plot (OLS Regression)

Beta can also be visualized as the slope of the regression line between asset returns and market returns.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(ret_market * 100, ret_ubs * 100, alpha=0.3, s=10, color='steelblue')

# Regression line
x_line = np.linspace(ret_market.min(), ret_market.max(), 100)
alpha_intercept = expected_return(ret_ubs) - ubs_beta * expected_return(ret_market)
y_line = alpha_intercept + ubs_beta * x_line
ax.plot(x_line * 100, y_line * 100, color='red', linewidth=2, label=f'Slope = beta = {ubs_beta:.3f}')

ax.set_xlabel('SMI Return (%)')
ax.set_ylabel('UBS Return (%)')
ax.set_title('UBS vs SMI — Beta Estimation', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

1. **Beta estimation**: $\beta = \text{Cov}(r_i, r_m) / \text{Var}(r_m)$ — systematic risk relative to the market
2. **CAPM formula**: $E[r_i] = r_f + \beta_i (E[r_m] - r_f)$ — required return for bearing systematic risk
3. **Security Market Line**: Plots CAPM expected return as a function of beta
4. **Alpha**: Difference between actual and CAPM-predicted return (mispricing or model error)